In [1]:
# ============================================================
# SECTION 1: Setup - run this script in a Google Colab Notebook cell
#
# !pip install pyngrok fastapi uvicorn python-multipart openai-whisper pydantic nest-asyncio transformers datasets torch pandas scikit-learn
#
# Paste the following code into the next cell and run it.
# ============================================================
!pip install pyngrok fastapi uvicorn python-multipart openai-whisper pydantic nest-asyncio transformers datasets torch pandas scikit-learn

# ============================================================
# SECTION 2: Imports
# ============================================================
import os
import sys
import re
import glob
import shutil
import zipfile
import json
import threading
import uuid
import nest_asyncio
from pyngrok import ngrok
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import FileResponse
from fastapi.encoders import jsonable_encoder
import uvicorn

import whisper
from transformers import AutoTokenizer, AutoConfig, Trainer, TrainingArguments, AutoModel
from datasets import Dataset, ClassLabel
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    mean_squared_error,
    mean_absolute_error,
    classification_report,
)
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd

nest_asyncio.apply()


# ============================================================
# SECTION 3: stdout / stderr Catcher for Live Logs
#
# Wraps stdout/stderr so every print() (including tqdm/Trainer
# logging) is also buffered into `catcher.logs`, which the
# /status/{task_id} endpoint drains and returns to the frontend
# as a live log stream.
# ============================================================
class LogCatcher:
    def __init__(self):
        self.logs = []
        self.original_stdout = sys.stdout
        self.original_stderr = sys.stderr

    def write(self, text):
        if text.strip():
            self.logs.append(text.strip())
        self.original_stdout.write(text)

    def flush(self):
        self.original_stdout.flush()

    def isatty(self):
        return False

catcher = LogCatcher()
sys.stdout = catcher
sys.stderr = catcher

tasks = {} # Store task state: {"status": "running|completed|failed", "result_file": str, "result_data": any}


# ============================================================
# SECTION 4: Transformer Classifier Logic
#   - DualHeadRoBERTa: backbone + two heads
#       - event_classifier: predicts the event type (try, penalty, ...)
#       - highlight_scorer: predicts a 0-1 "how exciting is this" score
#   - FocalLoss / WeightedMSELoss: losses for each head
#   - CustomTrainer: switches between the two losses via `stage`
# ============================================================

from transformers.modeling_outputs import ModelOutput
from dataclasses import dataclass
from typing import Optional
import torch

@dataclass
class DualHeadRoBERTaOutput(ModelOutput):
    loss: Optional[torch.FloatTensor] = None
    logits: torch.FloatTensor = None
    highlight_scores: torch.FloatTensor = None

class DualHeadRoBERTa(nn.Module):
    def __init__(self, model_name, num_labels, id2label, label2id):
        super().__init__()
        self.config = AutoModel.from_pretrained(model_name).config
        self.config.num_labels = num_labels
        self.config.id2label = id2label
        self.config.label2id = label2id

        self.roberta = AutoModel.from_pretrained(model_name)

        # Dual Heads
        self.event_classifier = nn.Linear(self.config.hidden_size, num_labels)
        self.highlight_scorer = nn.Linear(self.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask, labels=None, highlight_labels=None, **kwargs):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = getattr(outputs, "pooler_output", None)
        if pooled_output is None:
            pooled_output = outputs.last_hidden_state[:, 0, :]

        event_logits = self.event_classifier(pooled_output)
        highlight_score = torch.sigmoid(self.highlight_scorer(pooled_output))

        return DualHeadRoBERTaOutput(logits=event_logits, highlight_scores=highlight_score)


class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        if self.reduction == 'mean': return focal_loss.mean()
        elif self.reduction == 'sum': return focal_loss.sum()
        return focal_loss

class WeightedMSELoss(nn.Module):
    def __init__(self, base_weight=1.0, high_score_weight=5.0, threshold=0.01):
        super().__init__()
        self.base_weight = base_weight
        self.high_score_weight = high_score_weight
        self.threshold = threshold

    def forward(self, inputs, targets):
        mse = F.mse_loss(inputs, targets, reduction='none')
        weights = torch.where(targets > self.threshold, self.high_score_weight, self.base_weight)
        return (mse * weights).mean()


class CustomTrainer(Trainer):
    def __init__(self, *args, class_weights=None, stage=1, **kwargs):
        super().__init__(*args, **kwargs)
        self.stage = stage
        self.loss_fct_events = FocalLoss(alpha=class_weights, gamma=2.0)
        self.loss_fct_highlights = WeightedMSELoss(base_weight=1.0, high_score_weight=5.0, threshold=0.01)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        highlight_labels = inputs.pop("highlight_labels")

        outputs = model(**inputs)
        logits = outputs.logits
        highlight_scores = outputs.highlight_scores

        if self.stage == 1:
            if self.loss_fct_events.alpha is not None and self.loss_fct_events.alpha.device != logits.device:
                self.loss_fct_events.alpha = self.loss_fct_events.alpha.to(logits.device)
            loss = self.loss_fct_events(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        else:
            loss = self.loss_fct_highlights(highlight_scores.view(-1), highlight_labels.view(-1).to(highlight_scores.dtype))

        return (loss, outputs) if return_outputs else loss


# ============================================================
# SECTION 5: Sliding-Window Context Config
#
# WINDOW_SIZE=5 concatenates each row with its 2 neighbors on either
# side into one training sample, using the CENTER row's label and
# highlight_score as the target - commentary describing an event
# (e.g. a try) usually builds across several consecutive lines, so
# this gives the model that surrounding context instead of judging
# one isolated line. Persisted into the saved model's config so
# inference code can read it back instead of hardcoding it.
# ============================================================
WINDOW_SIZE = 5
MAX_LENGTH = 256
WINDOW_HALF = WINDOW_SIZE // 2


# ============================================================
# SECTION 6: Evaluation Metrics
#
# Reports both weighted and macro averages for precision/recall/f1.
# Weighted is dominated by normal_play (the majority class), so a
# model can look good on it while doing badly on rare events. Macro
# treats every class equally, which is why Stage 1's best checkpoint
# is selected by f1_macro instead of the weighted f1.
#
# The model always returns both heads' outputs, but only one head is
# actually being trained per stage - the other one's weights are
# frozen (or not yet trained), so its predictions are meaningless
# noise for that stage. Reporting classification metrics on a frozen
# classifier (Stage 2) or MSE/MAE on an untrained regressor (Stage 1)
# is misleading, so make_compute_metrics only computes the metrics
# that correspond to the stage actually being trained.
# ============================================================
def make_compute_metrics(stage):
    def compute_metrics(eval_pred):
        (event_logits, highlight_preds) = eval_pred.predictions
        (event_labels, highlight_labels) = eval_pred.label_ids

        if stage == 1:
            event_preds = event_logits.argmax(axis=-1)
            accuracy = accuracy_score(event_labels, event_preds)
            precision, recall, f1, _ = precision_recall_fscore_support(
                event_labels, event_preds, average="weighted", zero_division=0
            )
            precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
                event_labels, event_preds, average="macro", zero_division=0
            )
            return {
                "accuracy": accuracy,
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "precision_macro": precision_macro,
                "recall_macro": recall_macro,
                "f1_macro": f1_macro,
            }
        else:
            highlight_preds = highlight_preds.reshape(-1)
            highlight_labels = highlight_labels.reshape(-1)
            mse = mean_squared_error(highlight_labels, highlight_preds)
            mae = mean_absolute_error(highlight_labels, highlight_preds)
            return {
                "mse": mse,
                "mae": mae,
            }

    return compute_metrics


# ============================================================
# SECTION 7: Data Preprocessing Helpers
#
# Turns uploaded CSVs into windowed training rows, computes
# per-class loss weights, and logs a per-class breakdown of the
# best Stage 1 checkpoint. Used by run_train() below.
# ============================================================
def build_windowed_dataframe(csv_files, label2id):
    """
    Builds sliding-window training rows per CSV file, BEFORE concatenating
    them together, so a window can never bridge two different uploaded
    matches. The first and last WINDOW_HALF rows of each file are dropped
    rather than padded with a shorter window, so every training sample has
    full context.
    """
    windowed_dfs = []
    for f in csv_files:
        d = pd.read_csv(f)
        if 'event_class' in d.columns:
            d['label'] = d['event_class'].apply(lambda x: label2id.get(str(x).strip(), 0) if pd.notna(x) else 0)
        elif 'event' in d.columns:
            d['label'] = d['event'].apply(lambda x: label2id.get(str(x).strip(), 0) if pd.notna(x) else 0)

        if 'highlight_score' in d.columns:
            d['highlight_label'] = pd.to_numeric(d['highlight_score'], errors='coerce').fillna(0.0)
        else:
            d['highlight_label'] = 0.0

        d = d.dropna(subset=['text']).reset_index(drop=True)
        texts = d['text'].tolist()

        if len(texts) <= 2 * WINDOW_HALF:
            print(f"Skipping {f}: only {len(texts)} rows, need more than {2 * WINDOW_HALF} for window size {WINDOW_SIZE}")
            continue

        rows = []
        for i in range(WINDOW_HALF, len(texts) - WINDOW_HALF):
            window_text = " ".join(texts[i - WINDOW_HALF : i + WINDOW_HALF + 1])
            rows.append({
                'text': window_text,
                'label': d.loc[i, 'label'],
                'highlight_label': d.loc[i, 'highlight_label'],
            })
        windowed_dfs.append(pd.DataFrame(rows))

    return pd.concat(windowed_dfs, ignore_index=True)

def compute_class_weights(df_balanced, num_labels):
    total_samples = len(df_balanced)
    class_counts = df_balanced['label'].value_counts().to_dict()
    weights = []
    for i in range(num_labels):
        count = class_counts.get(i, 0)
        w = total_samples / (num_labels * count) if count > 0 else 1.0
        # FocalLoss handles class imbalance automatically.
        # Do NOT artificially down-weight normal_play here — it causes the model
        # to treat false positives as acceptable, destroying precision.
        weights.append(w)
    return torch.tensor(weights, dtype=torch.float)

def log_stage1_classification_report(trainer_stage1, test_dataset, id2label, num_labels):
    """Per-class breakdown of the best Stage 1 checkpoint, printed so it
    shows up in the live log stream the frontend polls."""
    pred_output = trainer_stage1.predict(test_dataset)
    event_logits, _ = pred_output.predictions
    event_labels, _ = pred_output.label_ids
    event_preds = event_logits.argmax(axis=-1)
    target_names = [id2label[i] for i in range(num_labels)]
    print("Stage 1 per-class report on held-out test split:")
    print(classification_report(
        event_labels, event_preds, labels=list(range(num_labels)), target_names=target_names, zero_division=0
    ))


# ============================================================
# SECTION 7B: Lexicon Model (Evaluation)
#
# Ported verbatim from commentary_analysis_system/src/models/lexicon_model.py
# (same logic colab testing/colab_evaluation.py already mirrors) - used by
# the /evaluate endpoint below to score "Lexicon Only" and "Hybrid Model"
# predictions.
# ============================================================
class LexiconModel:
    def __init__(self, config_path):
        with open(config_path, "r") as f:
            self.config = json.load(f)

    def _count_matches_weighted(self, text: str, terms: list[dict]) -> float:
        score = 0.0
        negations = [
            "no", "not", "missed", "missed the",
            "almost", "nearly", "just", "prevented",
            "stopped", "denied", "failed", "attempt",
            "trying to", "going for", "couldn't", "could not",
            "short of", "held up", "short",
        ]
        for term_obj in terms:
            if isinstance(term_obj, str):
                term, weight = term_obj, 1.0
            else:
                term = term_obj.get("text", "")
                weight = float(term_obj.get("weight", 1.0))
            if not term:
                continue

            if "\\" in term:
                pattern = term
            else:
                escaped_term = re.escape(term)
                pattern = rf"\b{escaped_term}(?:s|ed|ing|d)?\b"

            for match in re.finditer(pattern, text, re.IGNORECASE):
                start_idx = match.start()
                preceding_text = text[max(0, start_idx - 15):start_idx].lower()
                is_negated = any(preceding_text.strip().endswith(neg) for neg in negations)
                if not is_negated:
                    score += weight
        return score

    def generate_features(self, text: str) -> dict:
        text = text.lower()
        features = {}
        for category in self.config.get("categories", []):
            cat_id = category["id"]
            features[cat_id] = self._count_matches_weighted(text, category.get("terms", []))
        return features

    def score_chunk(self, text: str) -> float:
        feats = self.generate_features(text)
        raw_score = 0.0
        for category in self.config.get("categories", []):
            cat_id = category["id"]
            weight = category.get("weight", 0.0)
            count = feats.get(cat_id, 0)
            scaled_count = float(count) * 100.0
            raw_score += min(1.0, scaled_count) * weight
        return min(raw_score, 1.0)


# ============================================================
# SECTION 7C: Evaluation model loader + sliding-window builder
#
# Self-contained: reads a saved checkpoint directory in the same layout
# run_train() writes to best_dir/ (config.json, training_meta.json,
# pytorch_model.bin or model.safetensors, tokenizer files). Works whether
# the checkpoint was just trained in this session or uploaded fresh
# alongside the evaluation datasets.
# ============================================================
def build_context_windows(texts: list[str], window_size: int) -> list[str]:
    if window_size <= 1:
        return list(texts)
    half = window_size // 2
    n = len(texts)
    return [
        " ".join(texts[max(0, i - half): min(n, i + half + 1)])
        for i in range(n)
    ]


class EvalModel:
    def __init__(self, model_dir):
        config = AutoConfig.from_pretrained(model_dir)
        self.num_labels = config.num_labels
        self.id2label = {int(k): v for k, v in config.id2label.items()}
        self.label2id = {k: int(v) for k, v in config.label2id.items()}

        self.window_size = getattr(config, "window_size", 1)
        self.tokenizer_max_length = getattr(config, "tokenizer_max_length", MAX_LENGTH)

        meta_path = os.path.join(model_dir, "training_meta.json")
        if os.path.exists(meta_path):
            with open(meta_path, "r") as f:
                meta = json.load(f)
            self.window_size = meta.get("window_size", self.window_size)
            self.tokenizer_max_length = meta.get("tokenizer_max_length", self.tokenizer_max_length)

        base_model_name = getattr(config, "_name_or_path", "roberta-base")
        self.model = DualHeadRoBERTa(base_model_name, self.num_labels, self.id2label, self.label2id)

        bin_path = os.path.join(model_dir, "pytorch_model.bin")
        safetensors_path = os.path.join(model_dir, "model.safetensors")
        if os.path.exists(bin_path):
            state_dict = torch.load(bin_path, map_location="cpu")
        elif os.path.exists(safetensors_path):
            from safetensors.torch import load_file as load_safetensors
            state_dict = load_safetensors(safetensors_path)
        else:
            raise FileNotFoundError(f"No pytorch_model.bin or model.safetensors found in {model_dir}")
        self.model.load_state_dict(state_dict)

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval()

        self.tokenizer = AutoTokenizer.from_pretrained(model_dir)

    def predict_probs(self, text_list, batch_size=32):
        """Returns (event_probs, highlight_scores) - one list of softmax
        probs and one highlight-score float per input text."""
        all_event_probs, all_highlight_scores = [], []
        for i in range(0, len(text_list), batch_size):
            batch_texts = text_list[i:i + batch_size]
            inputs = self.tokenizer(
                batch_texts, padding=True, truncation=True,
                max_length=self.tokenizer_max_length, return_tensors="pt",
            )
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
            with torch.no_grad():
                outputs = self.model(**inputs)
                probs = torch.softmax(outputs.logits, dim=-1)
                all_event_probs.extend(probs.tolist())
                all_highlight_scores.extend(outputs.highlight_scores.view(-1).tolist())
        return all_event_probs, all_highlight_scores


# ============================================================
# SECTION 7D: Hybrid combination + evaluation metrics
#
# Ported formula-for-formula from colab testing/colab_evaluation.py, which
# itself mirrors commentary_analysis_system/src/models/hybrid_model.py.
# Weights are passed in per-request (the caller reads them from
# data/config/settings.json) rather than hardcoded, so this can't drift
# from the production hybrid formula the way a hardcoded copy would.
# ============================================================
APPROACH_NAMES = ["ML Model Only", "Lexicon Only", "Hybrid Model"]


def run_all_approaches(texts, id2label, model: EvalModel, lexicon: LexiconModel,
                        highlight_lexicon_weight, event_lexicon_weight, roberta_weight=1.0):
    windowed_texts = build_context_windows(texts, model.window_size)
    roberta_event_probs, roberta_highlight_scores = model.predict_probs(windowed_texts)

    ml_pred_event_id, ml_highlight_score = [], []
    lex_pred_event_id, lex_score = [], []
    hybrid_pred_event_id, hybrid_score_list = [], []

    for text, r_probs, r_h_score in zip(texts, roberta_event_probs, roberta_highlight_scores):
        # --- ML Model Only ---
        ml_event_id = max(range(len(r_probs)), key=lambda i: r_probs[i])
        ml_pred_event_id.append(ml_event_id)
        ml_highlight_score.append(r_h_score)

        # --- Lexicon Only ---
        l_prob = lexicon.score_chunk(text)
        lexicon_features = lexicon.generate_features(text)
        max_lex_score, lex_event_id = -1.0, 0
        for i in range(len(r_probs)):
            event_name = id2label.get(i, "normal_play")
            l_score = lexicon_features.get(event_name, 0.0)
            if l_score > max_lex_score:
                max_lex_score, lex_event_id = l_score, i
        lex_pred_event_id.append(lex_event_id)
        lex_score.append(l_prob)

        # --- Hybrid ---
        hybrid_score = min(1.0, (r_h_score * roberta_weight) + (l_prob * highlight_lexicon_weight))
        hybrid_score_list.append(hybrid_score)

        max_hybrid_prob, hyb_event_id = -1.0, 0
        for i, p in enumerate(r_probs):
            event_name = id2label.get(i, "normal_play")
            l_prob_for_event = lexicon_features.get(event_name, 0.0)
            l_prob_normalized = min(1.0, float(l_prob_for_event) * 100.0)
            event_hybrid_prob = (p * roberta_weight) + (l_prob_normalized * event_lexicon_weight)
            if event_hybrid_prob > max_hybrid_prob:
                max_hybrid_prob, hyb_event_id = event_hybrid_prob, i
        hybrid_pred_event_id.append(hyb_event_id)

    return {
        "ML Model Only": {"event_id": ml_pred_event_id, "score": ml_highlight_score},
        "Lexicon Only": {"event_id": lex_pred_event_id, "score": lex_score},
        "Hybrid Model": {"event_id": hybrid_pred_event_id, "score": hybrid_score_list},
    }


def eval_load_dataset_csv(csv_path, label2id):
    """Handles both current (event_class/highlight_score) and older
    (event/score) column names, same fallback used by build_windowed_dataframe()
    above and the local pipeline's evaluator.py: missing/blank event -> normal_play."""
    df = pd.read_csv(csv_path)
    df = df.dropna(subset=["text"]).reset_index(drop=True)

    event_col = "event_class" if "event_class" in df.columns else "event"
    score_col = "highlight_score" if "highlight_score" in df.columns else "score"

    df["true_event_id"] = df[event_col].apply(
        lambda x: label2id.get(str(x).strip(), 0) if pd.notna(x) and str(x).strip() else 0
    )
    df["true_score"] = pd.to_numeric(df[score_col], errors="coerce").fillna(0.0)
    return df


def compute_classification_metrics(y_true, y_pred):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    return {"precision": float(precision), "recall": float(recall), "f1": float(f1)}


def compute_per_class_metrics(y_true, y_pred, id2label):
    labels = sorted(id2label.keys())
    target_names = [id2label[i] for i in labels]
    report = classification_report(
        y_true, y_pred, labels=labels, target_names=target_names,
        output_dict=True, zero_division=0,
    )
    return {
        name: {
            "precision": float(report[name]["precision"]),
            "recall": float(report[name]["recall"]),
            "f1": float(report[name]["f1-score"]),
            "support": int(report[name]["support"]),
        }
        for name in target_names
    }


def compute_regression_metrics(y_true, y_pred):
    return {
        "mse": float(mean_squared_error(y_true, y_pred)),
        "mae": float(mean_absolute_error(y_true, y_pred)),
    }


def print_metrics_table(title, metrics_by_approach, keys):
    print(f"\n--- {title} ---")
    header = f"{'Approach':<16}" + "".join(f"{k:>16}" for k in keys)
    print(header)
    print("-" * len(header))
    for approach, metrics in metrics_by_approach.items():
        row = f"{approach:<16}" + "".join(f"{metrics[k]:>16.4f}" for k in keys)
        print(row)


# ============================================================
# SECTION 8: FastAPI Application & Whisper Model
# ============================================================
app = FastAPI()

print("Loading Whisper model (medium) ...")
whisper_model = whisper.load_model("medium")
print("Whisper model loaded!")


# ============================================================
# SECTION 9: Transcription Endpoint
# ============================================================
def run_transcribe(task_id, file_location):
    try:
        print(f"Task {task_id}: Starting Whisper Transcription...")
        initial_prompt = (
            "Rugby union commentary. Teams: Ireland, South Africa, England, France, New Zealand, Australia, Wales, Scotland. "
            "Rugby terms: try, conversion, penalty goal, drop goal, scrum, lineout, ruck, maul, offload, tackle, breakdown, turnover, "
            "sin bin, yellow card, red card, TMO, knock on, forward pass, offside, high tackle, line break, grubber kick, box kick."
        )

        result = whisper_model.transcribe(
            file_location,
            language="en",
            initial_prompt=initial_prompt,
            condition_on_previous_text=True,
            compression_ratio_threshold=2.4,
            no_speech_threshold=0.6,
            word_timestamps=True,
            verbose=True # Forces Whisper to print to stdout so we can catch it!
        )

        os.remove(file_location)
        # Set result_data before status: a poller reading "status: completed" must
        # never observe a dict where result_data is still missing (see predict_only.py
        # bug where a completed-but-dataless response wrote a null transcript).
        tasks[task_id]["result_data"] = jsonable_encoder(result["segments"])
        tasks[task_id]["status"] = "completed"
        print(f"Task {task_id}: Transcription completed successfully.")
    except Exception as e:
        print(f"Task {task_id} failed: {str(e)}")
        tasks[task_id]["status"] = "failed"

@app.post("/transcribe")
async def transcribe_audio(file: UploadFile = File(...)):
    task_id = str(uuid.uuid4())
    catcher.logs = [] # Reset logs for new task
    tasks[task_id] = {"status": "running"}

    file_location = f"/content/{file.filename}"
    with open(file_location, "wb") as buffer:
        shutil.copyfileobj(file.file, buffer)

    threading.Thread(target=run_transcribe, args=(task_id, file_location)).start()
    return {"task_id": task_id}


# ============================================================
# SECTION 10: Training Endpoint
#
# Runs the Stage 1 (event classifier) + Stage 2 (highlight scorer)
# training pipeline on the uploaded CSVs, then zips the best model
# so the frontend can download it for local predictions.
# ============================================================
def run_train(task_id, zip_path, model_name):
    try:
        print(f"Task {task_id}: Starting Training for {model_name}...")
        extract_dir = f"/content/training_data_{task_id}"
        output_dir = f"/content/model_output_{model_name.replace('/', '_')}"

        os.makedirs(extract_dir, exist_ok=True)
        os.makedirs(output_dir, exist_ok=True)

        # Only extract the CSV data sent from the frontend (~1MB, not the full model)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)

        # Read events.json
        events_path = os.path.join(extract_dir, "events.json")
        events = ["highlight"]
        if os.path.exists(events_path):
            with open(events_path, 'r') as f:
                events = json.load(f)

        label2id = {"normal_play": 0}
        id2label = {0: "normal_play"}
        for i, evt in enumerate(events, start=1):
            label2id[evt] = i
            id2label[i] = evt

        num_labels = len(label2id)

        print(f"Starting fresh training from '{model_name}'.")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = DualHeadRoBERTa(
            model_name, num_labels=num_labels, id2label=id2label, label2id=label2id
        )

        csv_files = glob.glob(os.path.join(extract_dir, "**", "*.csv"), recursive=True)
        df = build_windowed_dataframe(csv_files, label2id)
        df_balanced = df.sample(frac=1, random_state=42).reset_index(drop=True)
        class_weights = compute_class_weights(df_balanced, num_labels)

        # stratify_by_column keeps every event class represented in both the
        # train and test split, proportional to its overall frequency -
        # without it, a plain random split can leave a rare class with zero
        # test rows. train_test_split requires the stratify column to be a
        # ClassLabel feature, hence the cast.
        dataset = Dataset.from_pandas(df_balanced)
        dataset = dataset.cast_column(
            'label', ClassLabel(num_classes=num_labels, names=[id2label[i] for i in range(num_labels)])
        )

        def tokenize_func(examples):
            tokenized = tokenizer(examples['text'], padding="max_length", truncation=True, max_length=MAX_LENGTH)
            tokenized['labels'] = examples['label']
            tokenized['highlight_labels'] = examples['highlight_label']
            return tokenized

        tokenized_datasets = dataset.map(tokenize_func, batched=True)
        split_datasets = tokenized_datasets.train_test_split(test_size=0.2, seed=42, stratify_by_column='label')

        def get_training_args(output_dir_suffix, epochs=5, metric_for_best_model="loss", greater_is_better=None):
            return TrainingArguments(
                output_dir=f"{output_dir}/{output_dir_suffix}",
                num_train_epochs=epochs,
                per_device_train_batch_size=16,
                per_device_eval_batch_size=16,
                eval_strategy="epoch",
                logging_strategy="steps",
                logging_steps=5,
                save_strategy="epoch",
                load_best_model_at_end=True,
                metric_for_best_model=metric_for_best_model,
                greater_is_better=greater_is_better,
                disable_tqdm=False, # We want tqdm so we can catch it in stdout!
                learning_rate=2e-5,
                warmup_ratio=0.1
            )

        print("Starting Stage 1: Training Event Classifier...")
        # metric_for_best_model="f1_macro" makes load_best_model_at_end restore
        # the epoch that was actually best at classifying events (weighted
        # across all classes equally), not just the epoch with the lowest loss.
        trainer_stage1 = CustomTrainer(
            model=model,
            args=get_training_args("stage1", epochs=15, metric_for_best_model="f1_macro", greater_is_better=True),
            train_dataset=split_datasets['train'],
            eval_dataset=split_datasets['test'],
            class_weights=class_weights,
            stage=1,
            compute_metrics=make_compute_metrics(1),
        )
        trainer_stage1.train()

        log_stage1_classification_report(trainer_stage1, split_datasets['test'], id2label, num_labels)

        print("Starting Stage 2: Training Highlight Scorer...")
        # Freeze backbone and event classifier for Stage 2
        for name, param in model.named_parameters():
            if 'highlight_scorer' not in name:
                param.requires_grad = False

        # metric_for_best_model="mse" + greater_is_better=False restores the
        # epoch with the lowest highlight-score error, same reasoning as
        # Stage 1's f1_macro choice.
        trainer_stage2 = CustomTrainer(
            model=model,
            args=get_training_args("stage2", epochs=5, metric_for_best_model="mse", greater_is_better=False),
            train_dataset=split_datasets['train'],
            eval_dataset=split_datasets['test'],
            class_weights=class_weights,
            stage=2,
            compute_metrics=make_compute_metrics(2),
        )
        trainer_stage2.train()

        # Unfreeze after training
        for param in model.parameters():
            param.requires_grad = True

        # `model` already holds the best checkpoint's weights, not the last
        # epoch's: load_best_model_at_end restores the best-f1_macro epoch
        # after trainer_stage1.train() and the best-mse epoch after
        # trainer_stage2.train(). So this is already the best event
        # classifier + best highlight scorer, combined into one model.
        #
        # window_size/tokenizer_max_length are stashed on the config so
        # inference code can read back how this model expects its input to
        # be windowed, instead of hardcoding it separately.
        model.config.window_size = WINDOW_SIZE
        model.config.tokenizer_max_length = MAX_LENGTH

        # Zip the best model so the frontend can download it locally for predictions.
        best_dir = os.path.join(output_dir, "best")
        os.makedirs(best_dir, exist_ok=True)
        torch.save(model.state_dict(), os.path.join(best_dir, "pytorch_model.bin"))
        model.config.save_pretrained(best_dir)
        tokenizer.save_pretrained(best_dir)

        # Also write window_size/tokenizer_max_length as a plain sidecar file,
        # not just on model.config. transformers' config serialization is not
        # guaranteed to round-trip arbitrary custom attributes across every
        # version - if it silently drops them, inference falls back to
        # window_size=1 (unwindowed) while the model was actually trained on
        # WINDOW_SIZE-chunk context, which degrades predictions badly. This
        # file is unambiguous and always readable regardless of that.
        with open(os.path.join(best_dir, "training_meta.json"), "w") as f:
            json.dump({"window_size": WINDOW_SIZE, "tokenizer_max_length": MAX_LENGTH}, f)

        out_zip = f"/content/{model_name.replace('/', '_')}_trained.zip"
        shutil.make_archive(out_zip.replace('.zip', ''), 'zip', best_dir)

        print(f"Task {task_id}: Training complete! Model zipped to {out_zip}")
        tasks[task_id]["status"] = "completed"
        tasks[task_id]["result_file"] = out_zip
    except Exception as e:
        import traceback
        print(f"Task {task_id} failed:")
        traceback.print_exc()
        tasks[task_id]["status"] = "failed"

@app.post("/train")
async def train_model(file: UploadFile = File(...), model_name: str = Form(...)):
    task_id = str(uuid.uuid4())
    catcher.logs = [] # Reset logs for new task
    tasks[task_id] = {"status": "running"}

    zip_path = f"/content/upload_{task_id}.zip"
    with open(zip_path, "wb") as buffer:
        shutil.copyfileobj(file.file, buffer)

    threading.Thread(target=run_train, args=(task_id, zip_path, model_name)).start()
    return {"task_id": task_id}


# ============================================================
# SECTION 10B: Evaluation Endpoint
#
# Runs all 3 prediction approaches (ML Model Only / Lexicon Only / Hybrid
# Model) directly against the uploaded dataset CSVs, using the checkpoint
# uploaded alongside them (under model/ in the same zip) - i.e. real
# inference on this Colab GPU, not a precomputed local prediction file.
# This is what replaces the local evaluation pipeline (04_evaluate.py's
# old hand-rolled, macro-only, normal_play-excluded metrics), which was
# producing wrong numbers.
#
# Only the pooled/combined result across every uploaded CSV is returned -
# no per-match breakdown and no macro-average fields, matching what the
# frontend's "Final Evaluation Metrics" panel actually displays.
# ============================================================
_drive_mounted = False

def _ensure_drive_mounted():
    """Mounted lazily (only the first evaluation that actually needs Drive
    triggers the interactive auth flow), not at server startup - keeps
    transcribe/train-only sessions free of an unnecessary Drive prompt."""
    global _drive_mounted
    if not _drive_mounted:
        from google.colab import drive
        drive.mount("/content/drive")
        _drive_mounted = True


def resolve_model_dir(model_name, model_drive_path, extract_dir):
    """3-tier checkpoint resolution, cheapest first:
    1. Already resident in this Colab session (just trained via /train in
       the same runtime) - zero transfer.
    2. A zip on Google Drive - fast (Google-internal bandwidth), avoids
       re-uploading a 500MB+ checkpoint from the local machine over ngrok
       every single evaluation call.
    3. Uploaded alongside the evaluation data (under model/ in the zip) -
       slowest, but always works with no Drive setup required.
    """
    safe_name = model_name.replace("/", "_")

    session_model_dir = f"/content/model_output_{safe_name}/best"
    if os.path.exists(os.path.join(session_model_dir, "config.json")):
        print(f"Reusing checkpoint already trained in this session: {session_model_dir}")
        return session_model_dir

    uploaded_model_dir = os.path.join(extract_dir, "model")
    has_uploaded_model = os.path.exists(os.path.join(uploaded_model_dir, "config.json"))

    if model_drive_path or not has_uploaded_model:
        drive_relative_path = model_drive_path or f"colab_models/{safe_name}_model.zip"
        _ensure_drive_mounted()
        drive_zip_path = os.path.join("/content/drive/MyDrive", drive_relative_path.lstrip("/"))
        if os.path.exists(drive_zip_path):
            print(f"Loading checkpoint from Drive: {drive_zip_path}")
            drive_model_dir = os.path.join(extract_dir, "model_from_drive")
            os.makedirs(drive_model_dir, exist_ok=True)
            with zipfile.ZipFile(drive_zip_path, "r") as zf:
                zf.extractall(drive_model_dir)

            # The zip may extract straight into drive_model_dir, or into a single
            # subfolder (e.g. "best/") depending on how it was zipped - flatten
            # either layout so config.json ends up at drive_model_dir's root.
            if not os.path.exists(os.path.join(drive_model_dir, "config.json")):
                subdirs = [d for d in glob.glob(os.path.join(drive_model_dir, "*")) if os.path.isdir(d)]
                if len(subdirs) == 1 and os.path.exists(os.path.join(subdirs[0], "config.json")):
                    for item in os.listdir(subdirs[0]):
                        shutil.move(os.path.join(subdirs[0], item), drive_model_dir)
                    shutil.rmtree(subdirs[0])
            return drive_model_dir
        elif model_drive_path:
            raise FileNotFoundError(f"Model zip not found on Drive at {drive_zip_path}")

    if has_uploaded_model:
        print("Using checkpoint uploaded alongside the evaluation data.")
        return uploaded_model_dir

    safe_default = f"colab_models/{safe_name}_model.zip"
    raise FileNotFoundError(
        f"No checkpoint found for {model_name}: not resident in this Colab session, "
        f"nothing at Drive path 'MyDrive/{safe_default}', and none uploaded alongside "
        "the evaluation data. Either train this model first in this session, upload "
        f"its zip to that Drive path, or pass --upload-model from 04_evaluate.py."
    )


def run_evaluate(task_id, zip_path, model_name, highlight_lexicon_weight, event_lexicon_weight, model_drive_path=None):
    try:
        print(f"Task {task_id}: Starting evaluation for {model_name}...")
        extract_dir = f"/content/eval_data_{task_id}"
        if os.path.exists(extract_dir):
            shutil.rmtree(extract_dir)
        os.makedirs(extract_dir, exist_ok=True)

        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(extract_dir)

        csv_paths = sorted(glob.glob(os.path.join(extract_dir, "*.csv")))
        lexicon_path = os.path.join(extract_dir, "lexicon.json")

        if not csv_paths:
            raise FileNotFoundError("No dataset CSVs found in the uploaded zip.")
        if not os.path.exists(lexicon_path):
            raise FileNotFoundError("lexicon.json not found in the uploaded zip.")

        model_dir = resolve_model_dir(model_name, model_drive_path, extract_dir)

        print(f"Loading checkpoint for {model_name}...")
        eval_model = EvalModel(model_dir)
        print(f"Loaded model: num_labels={eval_model.num_labels}, window_size={eval_model.window_size}, "
              f"tokenizer_max_length={eval_model.tokenizer_max_length}")

        lexicon_model = LexiconModel(lexicon_path)
        print(f"Loaded lexicon with {len(lexicon_model.config.get('categories', []))} categories.")

        event_names = [eval_model.id2label[i] for i in sorted(eval_model.id2label.keys())]

        pooled_true_event, pooled_true_score = [], []
        pooled_pred_event = {name: [] for name in APPROACH_NAMES}
        pooled_pred_score = {name: [] for name in APPROACH_NAMES}

        for csv_path in csv_paths:
            match_name = os.path.splitext(os.path.basename(csv_path))[0]
            print(f"\nEvaluating: {match_name}")

            df = eval_load_dataset_csv(csv_path, eval_model.label2id)
            texts = df["text"].tolist()
            true_event = df["true_event_id"].tolist()
            true_score = df["true_score"].tolist()

            predictions = run_all_approaches(
                texts, eval_model.id2label, eval_model, lexicon_model,
                highlight_lexicon_weight, event_lexicon_weight,
            )

            for approach in APPROACH_NAMES:
                pooled_pred_event[approach].extend(predictions[approach]["event_id"])
                pooled_pred_score[approach].extend(predictions[approach]["score"])
            pooled_true_event.extend(true_event)
            pooled_true_score.extend(true_score)

        print(f"\n{'=' * 60}\nCOMBINED (all {len(csv_paths)} datasets pooled)\n{'=' * 60}")

        classification, per_class, regression = {}, {}, {}
        for approach in APPROACH_NAMES:
            classification[approach] = compute_classification_metrics(
                pooled_true_event, pooled_pred_event[approach]
            )
            per_class[approach] = compute_per_class_metrics(
                pooled_true_event, pooled_pred_event[approach], eval_model.id2label
            )
            regression[approach] = compute_regression_metrics(
                pooled_true_score, pooled_pred_score[approach]
            )

        print_metrics_table("Classification", classification, ["precision", "recall", "f1"])
        print_metrics_table("Regression", regression, ["mse", "mae"])

        print(f"Task {task_id}: Evaluation complete!")
        tasks[task_id]["result_data"] = {
            "Classification": classification,
            "Regression": regression,
            "PerClass": per_class,
        }
        tasks[task_id]["status"] = "completed"

        shutil.rmtree(extract_dir, ignore_errors=True)
        os.remove(zip_path)
    except Exception as e:
        import traceback
        print(f"Task {task_id} failed:")
        traceback.print_exc()
        tasks[task_id]["status"] = "failed"

@app.post("/evaluate")
async def evaluate_model(
    file: UploadFile = File(...),
    model_name: str = Form(...),
    highlight_lexicon_weight: float = Form(0.75),
    event_lexicon_weight: float = Form(0.35),
    model_drive_path: str = Form(None),
):
    task_id = str(uuid.uuid4())
    catcher.logs = [] # Reset logs for new task
    tasks[task_id] = {"status": "running"}

    zip_path = f"/content/eval_upload_{task_id}.zip"
    with open(zip_path, "wb") as buffer:
        shutil.copyfileobj(file.file, buffer)

    threading.Thread(
        target=run_evaluate,
        args=(task_id, zip_path, model_name, highlight_lexicon_weight, event_lexicon_weight, model_drive_path),
    ).start()
    return {"task_id": task_id}


# ============================================================
# SECTION 11: Status & Download Endpoints
# ============================================================
@app.get("/status/{task_id}")
async def get_status(task_id: str):
    if task_id not in tasks:
        return {"status": "error", "message": "Task not found"}

    # Drain logs (return logs and clear them so we don't send duplicates)
    current_logs = list(catcher.logs)
    catcher.logs.clear()

    return {
        "status": tasks[task_id]["status"],
        "logs": current_logs,
        "result_data": tasks[task_id].get("result_data")
    }

@app.get("/download/{task_id}")
async def download_result(task_id: str):
    if task_id not in tasks or tasks[task_id]["status"] != "completed":
        return {"error": "Not ready"}
    file_path = tasks[task_id].get("result_file")
    if not file_path or not os.path.exists(file_path):
        return {"error": "File not found"}
    return FileResponse(file_path, media_type="application/zip", filename=os.path.basename(file_path))


# ============================================================
# SECTION 12: Server Startup (ngrok tunnel + uvicorn)
# ============================================================
NGROK_TOKEN = "3HaGzT01qmCqQsQt9O4XeApdUo4_7hTA4NQeb6jCiHP3JagCx" # <-- PASTE YOUR NGROK TOKEN HERE
ngrok.set_auth_token(NGROK_TOKEN)
ngrok.kill()
public_url = ngrok.connect(8000).public_url

print(f"\n=======================================================")
print(f"✅ YOUR CLOUD API IS LIVE AT: {public_url}")
print(f"✅ Update your local system's .env file with this URL!")
print(f"=======================================================\n")

# Start Uvicorn in a background thread so it doesn't fight Jupyter's event loop
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, access_log=False)

threading.Thread(target=run_server, daemon=True).start()

# Keep the main thread alive so the Colab cell doesn't finish!
import time
while True:
    time.sleep(1)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 16.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=ad4d76b11825a7b3b7f7578612165d7cf30e830fc5751afa60601a61033b8c28
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper
Loading Whisper model (medium) ...
100%|#####################################| 1.42G/1.42G [00:20<00:00, 74.2MiB/s]
Whisper model loaded!

✅ YOUR CLOUD API IS LIVE AT: https://craftsman-aide-dust.ngrok-free.dev
✅ Update your local system's .env file with this URL!

INFO:     Started server process [2498]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
Task 565e18b

config.json:   0%|          | 0.00/474 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Stage 1 per-class report on held-out test split:
              precision    recall  f1-score   support

 normal_play       0.96      0.82      0.89       818
         try       0.52      0.93      0.67        15
   goal_kick       0.67      0.91      0.77        46
  card_event       1.00      1.00      1.00         3
       scrum       0.56      0.83      0.67        42
        maul       0.61      0.79      0.69        14
     lineout       0.42      0.65      0.51        40
    kick_off       0.49      0.91      0.63        22
  tmo_replay       0.67      0.93      0.78        15
     penalty       0.43      0.80      0.56        20

    accuracy                           0.83      1035
   macro avg       0.63      0.86      0.72      1035
weighted avg       0.88      0.83      0.84      1035

Starting Stage 2: Training Highlight Scorer...


Epoch,Training Loss,Validation Loss,Mse,Mae
1,0.161874,0.162612,0.091256,0.259070
2,0.123162,0.141311,0.075834,0.239574
3,0.135521,0.133945,0.073469,0.235903
4,0.094286,0.130591,0.072107,0.233766
5,0.111434,0.129629,0.071975,0.233431


Task 565e18b3-d05e-40db-b886-fff298eb9cfc: Training complete! Model zipped to /content/microsoft_deberta-base_trained.zip


KeyboardInterrupt: 